# Genie API Client
Programmatic interface to the Gripper Monitoring Genie Space (`01f1c491058d10f1a5fdbb95f67c1fdf`) via the Databricks REST API.

Supports both **Chat mode** (stateful conversation) and **Management** operations.

In [0]:
import requests, time

class GenieClient:
    """Thin wrapper around the Databricks Genie Agents REST API."""

    def __init__(self, space_id: str, host: str = None, token: str = None):
        if host is None:
            ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
            host = ctx.apiUrl().get()
            token = ctx.apiToken().get()
        self.space_id = space_id
        self._base = f"{host}/api/2.0/genie/spaces/{space_id}"
        self._headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

    # ------------------------------------------------------------------ #
    # Conversation lifecycle
    # ------------------------------------------------------------------ #
    def start_conversation(self, question: str) -> dict:
        """Start a new conversation and return the full initial message result."""
        r = requests.post(
            f"{self._base}/start-conversation",
            headers=self._headers,
            json={"content": question}
        )
        r.raise_for_status()
        payload = r.json()
        conv_id = payload["conversation"]["id"]
        msg_id  = payload["message"]["id"]
        return self._wait_for_message(conv_id, msg_id)

    def ask(self, conversation_id: str, question: str) -> dict:
        """Send a follow-up message in an existing conversation."""
        r = requests.post(
            f"{self._base}/conversations/{conversation_id}/messages",
            headers=self._headers,
            json={"content": question}
        )
        r.raise_for_status()
        msg_id = r.json()["id"]
        return self._wait_for_message(conversation_id, msg_id)

    # ------------------------------------------------------------------ #
    # Polling
    # ------------------------------------------------------------------ #
    def _wait_for_message(
        self, conversation_id: str, message_id: str,
        poll_interval: float = 2.0, timeout: int = 120
    ) -> dict:
        """Poll until the message reaches a terminal state, then return it."""
        url = f"{self._base}/conversations/{conversation_id}/messages/{message_id}"
        deadline = time.time() + timeout
        while time.time() < deadline:
            r = requests.get(url, headers=self._headers)
            r.raise_for_status()
            msg = r.json()
            status = msg.get("status", "")
            if status in ("COMPLETED", "FAILED", "CANCELLED"):
                msg["_conversation_id"] = conversation_id
                return msg
            time.sleep(poll_interval)
        raise TimeoutError(f"Message {message_id} did not complete within {timeout}s")

    # ------------------------------------------------------------------ #
    # Result extraction helpers
    # ------------------------------------------------------------------ #
    def get_sql(self, message: dict) -> str | None:
        """Extract the generated SQL from a completed message."""
        for attachment in message.get("attachments", []):
            if "query" in attachment:
                return attachment["query"].get("query")
        return None

    def get_text(self, message: dict) -> str | None:
        """Extract the natural-language summary from a completed message."""
        for attachment in message.get("attachments", []):
            if "text" in attachment:
                return attachment["text"].get("content")
        return None

    def get_query_result(self, message: dict) -> dict | None:
        """Fetch the query result rows for a completed message that ran SQL."""
        conv_id = message.get("_conversation_id")
        msg_id  = message.get("id")
        if not conv_id or not msg_id:
            return None
        r = requests.get(
            f"{self._base}/conversations/{conv_id}/messages/{msg_id}/query-result",
            headers=self._headers
        )
        if r.status_code == 404:
            return None
        r.raise_for_status()
        return r.json()

    # ------------------------------------------------------------------ #
    # Management helpers
    # ------------------------------------------------------------------ #
    def get_space(self) -> dict:
        """Return the Genie space configuration."""
        r = requests.get(self._base, headers=self._headers)
        r.raise_for_status()
        return r.json()

print("GenieClient ready")

In [0]:
SPACE_ID = "01f1c491058d10f1a5fdbb95f67c1fdf"

client = GenieClient(SPACE_ID)

# Start a new conversation
msg = client.start_conversation(
    "What is the average OEE per shift over the last 7 days?"
)

print("Status:", msg.get("status"))
print("\n--- Summary ---")
print(client.get_text(msg))
print("\n--- Generated SQL ---")
print(client.get_sql(msg))

In [0]:
result = client.get_query_result(msg)

if result:
    from pyspark.sql import Row
    schema = result.get("statement_response", {}).get("manifest", {}).get("schema", {}).get("columns", [])
    columns = [c["name"] for c in schema]
    rows    = result.get("statement_response", {}).get("result", {}).get("data_typed_array", [])
    
    data = []
    for row in rows:
        values = [v.get("str") for v in row["values"]]
        data.append(dict(zip(columns, values)))
    
    if data:
        df = spark.createDataFrame([Row(**r) for r in data])
        df.show(truncate=False)
    else:
        print("No rows returned.")
else:
    print("No query result (text-only answer).")

In [0]:
conv_id = msg["_conversation_id"]

follow_up = client.ask(conv_id, "Which shift has the highest defect rate?")

print("Status:", follow_up.get("status"))
print(client.get_text(follow_up))
print(client.get_sql(follow_up))